# 新闻标题能预测道琼斯指数涨跌吗？

**问题**：每天的世界新闻标题，真的能帮助预测道指（DJIA）第二天/当天的涨跌吗？如果有作用，它在统计上和经济上是真实的吗？

**用户**：一个刷到"AI 读新闻预测股市"广告的普通散户。

This project is a prototype and is not financial advice.

## 阶段 1：环境、数据清理、检查、切分、基准

### 1.0 导入与随机种子
为什么固定随机种子 42：之后的模型、bootstrap 都有随机性，固定种子才能让别人重跑得到一样的数字。

In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# 让 notebook 能 import 项目根目录下的 src/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import data_utils as du
from src.evaluation import evaluate, confusion_table

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA = ROOT / "data"
pd.set_option("display.max_colwidth", 120)

### 1.1 读取并清理新闻数据
- 原始标题很多长这样：`b'Georgia downs two Russian warplanes...'`，这是 Python 字节串被直接写进了 CSV，`b'` 和结尾的 `'` 不是标题内容，要去掉。
- 有少数格子是空的（NaN），换成空字符串，免得后面打情绪分时报错。
- `Date` 转成日期类型并排序，因为之后所有切分都按时间来。

In [ ]:
news_raw = pd.read_csv(DATA / "Combined_News_DJIA.csv")
news = du.load_news(DATA / "Combined_News_DJIA.csv")

print("行数、列数:", news.shape)
print("日期范围:", news["Date"].min().date(), "到", news["Date"].max().date())
print("每列标题的空值数（清理前，只显示非零的）:")
na = news_raw[du.HEADLINE_COLS].isna().sum()
print(na[na > 0])

has_prefix = news_raw[du.HEADLINE_COLS].apply(lambda s: s.astype(str).str.match(r'^b[\'"]')).sum().sum()
left = news[du.HEADLINE_COLS].apply(lambda s: s.str.match(r'^b[\'"]')).sum().sum()
print(f"清理前带 b'' 前缀的标题: {has_prefix}，清理后仍带前缀: {left}")

In [ ]:
# 清理前后对比，看看效果
print("清理前:", news_raw.loc[0, "Top1"])
print("清理后:", news.loc[0, "Top1"])

### 1.2 读取道指价格并检查 Label
数据集作者说 Label = 1 表示道指"上涨或持平"。我们用价格自己算一遍，看看和 Label 是否一致。
同时检查两种可能的定义：**收盘价对前一天收盘价**、**收盘价对当天开盘价**。哪种一致率接近 100%，就说明 Label 实际是按哪种定义做的。

In [ ]:
djia = du.load_djia(DATA / "DJIA_table.csv")
print("道指表:", djia.shape, djia["Date"].min().date(), "到", djia["Date"].max().date())

# 两张表的日期是否一一对应
only_news = set(news["Date"]) - set(djia["Date"])
only_djia = set(djia["Date"]) - set(news["Date"])
print("只在新闻表里的日期数:", len(only_news), "| 只在道指表里的日期数:", len(only_djia))

merged, rates = du.check_labels(news, djia)
pd.DataFrame(rates).T

In [ ]:
# 把不一致的日子列出来看看（用收盘对前一天收盘的定义）
mismatch = merged[merged["up_adjclose"].notna() & (merged["up_adjclose"] != merged["Label"])]
print("不一致的天数:", len(mismatch))
mismatch[["Date", "Label", "Open", "Close", "Adj Close", "up_adjclose", "up_open_close"]].head(15)

### 1.3 按日期切分
- 训练集：2008-08-08 到 2014-12-31
- 测试集：2015-01-02 到 2016-07-01

为什么按日期而不是随机打乱：现实中我们只能用过去预测未来。如果随机打乱，模型会"偷看"到未来的日子，结果会虚高。

In [ ]:
train, test = du.split_by_date(news)
assert train["Date"].max() < test["Date"].min()  # 确认训练集完全在测试集之前
assert len(train) + len(test) == len(news)        # 确认没有日子被漏掉

split_summary = pd.DataFrame({
    "起始": [train["Date"].min().date(), test["Date"].min().date()],
    "结束": [train["Date"].max().date(), test["Date"].max().date()],
    "天数": [len(train), len(test)],
    "涨的比例": [train["Label"].mean(), test["Label"].mean()],
}, index=["训练集", "测试集"])
split_summary

### 1.4 两个基准（只在测试集上算）
任何 AI 模型都必须先打败这两个"笨办法"，否则就没有价值：
1. **永远预测涨**：股市长期大多数日子在涨，所以这个准确率不会太低。
2. **跟前一天同方向**：检验"动量"——昨天涨今天也涨。测试集第一天用训练集最后一天的 Label。

指标：准确率为主；再看混淆矩阵和"跌"的召回率（真实下跌的日子里，猜中了几成）。永远预测涨的"跌"召回率必然是 0，这说明光看准确率会被骗。

In [ ]:
y_test = test["Label"].to_numpy()
preds = {
    "永远预测涨": du.baseline_always_up(test),
    "跟前一天同方向": du.baseline_previous_day(train, test),
}

results = pd.DataFrame([evaluate(y_test, p, name) for name, p in preds.items()]).set_index("model")
results

In [ ]:
for name, p in preds.items():
    print(f"\n=== {name} ===")
    print(confusion_table(y_test, p))

### 阶段 1 小结
（见上方输出；结论会在看完结果后补写。）